# Find a user collection and Explore it

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-10 : stage 1
- update : 2026-10-16 : stage 2
- last update : 2026-10-28 : display of the postISRCCD image with afwDisplay (matplotlib backend), RA increasing to the left

- `fichier bps_dp2cosmos.sh`

```bash
export VISITIDS=$(paste -sd, cosmos_small.txt)
export CCDIDS=$(paste -sd, ccdIds.txt)

release="w_2026_40"

echo "run BPS"
bps_submit.sh --interactive bps_dp2cosmos_parsl.yaml
```

- `fichier bps_dp2cosmos_parsl.yaml`
```
pipelineYaml: "${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit"

payload:
  runInit: true
  payloadName: dp2cosmos202610
  butlerConfig: main
  inCollection: LSSTCam/defaults
  output: u/${USER}/LSSTCam/runcosmos01_20261008_RunCosmosDemo/stage1
  dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND detector IN (${CCDIDS}) AND exposure IN (${VISITIDS}) AND visit IN (${VISITIDS})"
```

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graph : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import warnings

# Turn the deprecated Butler.collections FutureWarning into an error to catch outdated API usage early
warnings.filterwarnings(
    "error",
    category=FutureWarning,
    message=".*Butler.collections.*",
)

In [ ]:
# Butler: entry point to the data repository
from lsst.daf.butler import Butler

# Repository label (alias configured in the Rubin environment at the computing centre)
butler_url = "main"
butler = Butler(butler_url)

## Identify existing user collections

In [ ]:
from lsst.daf.butler import CollectionType

# User collections live under the u/<username>/ namespace
prefix = "u/dagoret/"

collections = list(butler.collections.query(f"{prefix}*"))

# Print the name of each collection found
for collection in collections:
    print(collection)

print(f"\nNombre de collections : {len(collections)}")


## 

### Choose the collection

In [ ]:
# Full name of the collection to explore (stage 1 here; use the stage 2 line below to inspect later outputs)
collection_path = (
    #stage 1 collection
    "u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/"
    "stage1/20261008T151954Z"
    #stage 2 collection
    #"u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage2/20261016T135502Z"
)

# Check that the chosen collection exists in the repository
exists = any(
    name == collection_path
    for name in butler.collections.query(collection_path)
)
print(f"exists = {exists} for {collection_path}")

if collection_path in butler.collections.query(collection_path):
    info = butler.collections.get_info(collection_path)
    print(f"Collection trouvée : {collection_path}")
    print(f"Type : {info.type}")
else:
    print(f"Collection inexistante : {collection_path}")

## Find datasets in my collection

In [ ]:
# Standard library, used below to count datasets per dataset type
import collections

### Total number of datasets

In [ ]:
# Query every dataset (all dataset types) stored in the collection
refs = butler.registry.queryDatasets(
    datasetType="*",
    collections=collection_path,
    findFirst=False,
)

refs = list(refs)

print(f"Total number of datasets referenced  : {len(refs)} in {collection_path}")

# Count the datasets per dataset type
counts = collections.Counter(
    ref.datasetType.name for ref in refs
)

for name, count in sorted(counts.items()):
    print(f"{name:40s} {count:8d}")

### Examine a number of references

In [ ]:
# Print the first references (dataset type and data ID) as a sanity check
for ref in refs[:20]:
    print(
        ref.datasetType.name,
        ref.dataId
    )

### Examine post_isr_image

In [ ]:
# List the data IDs of all post_isr_image datasets (one per exposure and detector)
for ref in butler.registry.queryDatasets(
    datasetType="post_isr_image",
    collections=collection_path,
    findFirst=False,
):
    print(ref.dataId)

Dans ta collection, il y a plusieurs catalogues liés aux étoiles. Pour le pipeline Rubin que tu as exécuté, les trois types les plus intéressants sont les suivants.


- Dataset type	    :->  Contenu / rôle 

- single_visit_star	: ->  Catalogue consolidé des étoiles détectées sur plusieurs visites, produit final de l'association des étoiles sur une visite unique.

- single_visit_star_detector :->	Catalogue des étoiles associé aux détecteurs individuels (179 datasets dans ta collection).

- isolated_star	    :-> Échantillons d'étoiles isolées, utilisés notamment pour la calibration photométrique et astrométrique.


- isolated_star_association	:-> Résultats de l'association des étoiles isolées entre observations.

  
- single_visit_psf_star	    :-> Étoiles sélectionnées pour la modélisation de la PSF, une sélection spécifique et non un catalogue stellaire général.


Je te conseille de commencer par single_visit_star, puis de regarder single_visit_star_detector si tu souhaites étudier les étoiles détectées sur chaque CCD.


### 1. Lire le catalogue single_visit_star

In [ ]:
# Query the single_visit_star catalogues of the collection
refs = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs))

for ref in refs:
    print(ref.dataId)

### 2. Charger le catalogue dans un DataFrame

In [ ]:
import pandas as pd

# Take the first catalogue reference and load it from the Butler
ref = refs[0]

catalog = butler.get(ref)

print(type(catalog))
print(len(catalog))
#print(catalog.schema)

In [ ]:
catalog

In [ ]:
# Convert the catalogue to a pandas DataFrame for easy inspection
df = catalog.to_pandas()

display(df.head())
print(df.columns.tolist())

### 3. Examiner les étoiles détectées par détecteur

Pour accéder aux 179 catalogues single_visit_star_detector :

In [ ]:
# Query the per-detector star catalogues
refs_detector = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs_detector))

catalog_detector = butler.get(refs_detector[0])

print(type(catalog_detector))


In [ ]:
# NOTE: this converts `catalog` (the visit-level table loaded above), not `catalog_detector`;
# use catalog_detector.to_pandas() to inspect the per-detector table
df_det = catalog.to_pandas()

display(df_det.head())
print(df_det.columns.tolist())

### 4 Image

#### 4.1. Sélectionner un détecteur commun aux deux datasets
Dans ton notebook, commence par afficher les dataId disponibles :

In [ ]:
import pandas as pd

# Query the post-ISR images and the per-detector star catalogues of the collection
refs_img = list(
    butler.registry.queryDatasets(
        datasetType="post_isr_image",
        collections=collection_path,
    )
)

refs_cat = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Images :", len(refs_img))
print("Catalogues :", len(refs_cat))

print("\nPremières images :")
for ref in refs_img[:5]:
    print(ref.dataId)

print("\nPremiers catalogues :")
for ref in refs_cat[:5]:
    print(ref.dataId)

Repère un couple qui possède le même visit et le même detector.
#### 4.2. Charger l'image et le catalogue
Une fois le dataId commun identifié, sélectionne les deux références correspondantes. Par exemple :

In [ ]:
print("IMAGE")
print(refs_img[0].dataId)
print("Dimensions :", list(refs_img[0].dataId.dimensions.names))

print("\nCATALOGUE")
print(refs_cat[0].dataId)
print("Dimensions :", list(refs_cat[0].dataId.dimensions.names))

In [ ]:
# Set of detectors available for each dataset type
detectors_img = {
    r.dataId["detector"]
    for r in refs_img
}

detectors_cat = {
    r.dataId["detector"]
    for r in refs_cat
}

# Keep only the detectors that have both an image and a catalogue
common_detectors = sorted(detectors_img & detectors_cat)

print("Nombre de détecteurs communs :", len(common_detectors))
print("Détecteurs :", common_detectors)

In [ ]:
# Detector to work on (must be one of the common detectors listed above)
detector_id = 47  # Replace with the chosen detector

# Select the post-ISR image reference for this detector
ref_img = next(
    r for r in refs_img
    if r.dataId["detector"] == detector_id
)

# Select the matching catalogue: same detector, and visit equal to the image exposure ID
ref_cat = next(
    r for r in refs_cat
    if r.dataId["detector"] == detector_id
    and r.dataId["visit"] == ref_img.dataId["exposure"]
)

# Load the post-ISR exposure and the star catalogue from the Butler
exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Image :", ref_img.dataId)
print("Catalogue :", ref_cat.dataId)
print("Dimensions image :", exposure.image.array.shape)
print("Type catalogue :", type(catalog))

#### 4.3 Charger les deux données

In [ ]:
exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Type image :", type(exposure))
print("Type catalogue :", type(catalog))
print("Dimensions image :", exposure.image.array.shape)

In [ ]:
# Show the catalogue columns (afw table schema or astropy table column names)
print(catalog.schema if hasattr(catalog, "schema") else catalog.colnames)

### 5. Display the postISRCCD image with `lsst.afw.display` (matplotlib backend)

- `lsst.afw.display` is the generic Rubin display interface. It can drive several backends, among them `firefly` and `matplotlib`.
- The `firefly` backend needs a reachable Firefly server (available on the Rubin Science Platform), which is normally **not** the case at CC-IN2P3.
- The `matplotlib` backend needs no server and works in a Jupyter notebook at CC-IN2P3, so it is the default here.
- The axes are flipped, using the WCS of the exposure, so that RA increases to the left (East left) and Dec upward (North up); a yellow E/N compass allows to check the orientation.
- To use Firefly elsewhere, set `BACKEND = "firefly"` in the next cell.

In [ ]:
# Render figures inline in the notebook (works in Jupyter at CC-IN2P3)
%matplotlib inline

import os

import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay

# Display backend: "matplotlib" (no server needed, default at CC-IN2P3)
# or "firefly" (needs a Firefly server, e.g. on the Rubin Science Platform)
BACKEND = "matplotlib"

# Set the backend used by every afwDisplay.Display object created afterwards
afwDisplay.setDefaultBackend(BACKEND)
print(f"afw display backend : {BACKEND}")

In [ ]:
# Sanity check on the post-ISR exposure loaded in section 4 (variables `ref_img` and `exposure`)
print("Data ID              :", ref_img.dataId)
print("Image shape (ny, nx) :", exposure.image.array.shape)
print("Bounding box         :", exposure.getBBox())

# Mask planes attached to the exposure (e.g. BAD, SAT, CR, ...), shown as overlays below
print("Mask planes          :", list(exposure.mask.getMaskPlaneDict().keys()))

In [ ]:
import numpy as np
import lsst.geom as geom

# Create the figure that will host the image (matplotlib backend)
fig = plt.figure(figsize=(10, 10))

# With the matplotlib backend the display is attached to the figure itself;
# with the firefly backend the frame is a simple integer identifier
frame = fig if BACKEND == "matplotlib" else 1
afw_display = afwDisplay.Display(frame=frame)

# Intensity stretch: asinh scaling with a zscale interval, well suited to sky-dominated CCD images
afw_display.scale("asinh", "zscale")

# Transparency of the mask plane overlay (0 = opaque, 100 = fully transparent)
afw_display.setMaskTransparency(70)

# Title identifying the exposure and the detector
title = (
    f"postISRCCD  exposure={ref_img.dataId['exposure']}  "
    f"detector={ref_img.dataId['detector']}"
)

# Draw the exposure (pixel data + mask planes) in the display
afw_display.mtv(exposure, title=title)

# --- Sky orientation: RA increasing to the left (East left), Dec increasing upward (North up) ---
# Uses the WCS of the exposure; only needed for the matplotlib backend
# (Firefly handles the sky orientation itself)
wcs = exposure.getWcs()
if BACKEND == "matplotlib" and wcs is not None:
    ax = fig.gca()

    # Pixel at the centre of the image and step used to measure the local WCS Jacobian
    center = exposure.getBBox().getCenter()
    xc, yc = center.getX(), center.getY()
    step = 100.0

    def sky_deg(x, y):
        # (RA, Dec) in degrees of a pixel position
        sp = wcs.pixelToSky(geom.Point2D(x, y))
        return sp.getRa().asDegrees(), sp.getDec().asDegrees()

    ra0, dec0 = sky_deg(xc, yc)
    ra_x, dec_x = sky_deg(xc + step, yc)
    ra_y, dec_y = sky_deg(xc, yc + step)

    def d_east(ra):
        # RA offset wrapped to [-180, 180) deg and projected on the sky (times cos Dec)
        return ((ra - ra0 + 180.0) % 360.0 - 180.0) * np.cos(np.radians(dec0))

    # Jacobian: rows = (East, North) offsets in deg, columns = (x, y) pixel steps
    jac = np.array([[d_east(ra_x), d_east(ra_y)],
                    [dec_x - dec0, dec_y - dec0]]) / step

    # Unit vectors, in pixel (x, y) space, pointing towards East and towards North
    inv = np.linalg.inv(jac)
    east_pix = inv[:, 0] / np.hypot(*inv[:, 0])
    north_pix = inv[:, 1] / np.hypot(*inv[:, 1])

    # Same vectors as seen on screen (sign change if an axis is already inverted)
    sx = -1.0 if ax.xaxis_inverted() else 1.0
    sy = -1.0 if ax.yaxis_inverted() else 1.0
    east_scr = np.array([sx * east_pix[0], sy * east_pix[1]])
    north_scr = np.array([sx * north_pix[0], sy * north_pix[1]])

    if abs(east_scr[0]) >= abs(east_scr[1]):
        # The x axis is roughly the East-West axis: flipping the axes is enough
        if east_scr[0] > 0:
            ax.invert_xaxis()  # East points right -> make RA increase to the left
        if north_scr[1] < 0:
            ax.invert_yaxis()  # North points down -> make Dec increase upward
    else:
        # The x axis is roughly the North-South axis (detector rotated by ~90 deg): flips are not enough
        print("WARNING: detector rotated by ~90 deg on the sky, axis flips cannot give East-left / North-up")

    # Compass drawn in data coordinates (it follows the axis flips): E and N arrows
    bbox = exposure.getBBox()
    base = np.array([bbox.getMinX() + 0.15 * bbox.getWidth(),
                     bbox.getMinY() + 0.15 * bbox.getHeight()])
    length = 0.1 * min(bbox.getWidth(), bbox.getHeight())
    for label, vec in (("E", east_pix), ("N", north_pix)):
        tip = base + length * vec
        ax.annotate("", xy=tip, xytext=base,
                    arrowprops=dict(arrowstyle="->", color="yellow", lw=2))
        ax.text(*(base + 1.25 * length * vec), label, color="yellow",
                ha="center", va="center", fontsize=14)

    print("East direction in pixel space (x, y)  :", np.round(east_pix, 3))
    print("North direction in pixel space (x, y) :", np.round(north_pix, 3))
elif BACKEND == "matplotlib":
    print("No WCS attached to this exposure: the raw pixel orientation is kept")

plt.show()

In [ ]:
# Save the figure as PNG and PDF (matplotlib backend only)
if BACKEND == "matplotlib":
    # Output directory, created if needed
    figdir = "figs_postISRCCD"
    os.makedirs(figdir, exist_ok=True)

    # Common file name stem built from the exposure and detector IDs
    stem = (
        f"postISRCCD_exposure{ref_img.dataId['exposure']}"
        f"_det{ref_img.dataId['detector']}"
    )

    for ext in ("png", "pdf"):
        outname = os.path.join(figdir, f"{stem}.{ext}")
        fig.savefig(outname, dpi=150, bbox_inches="tight")
        print(f"Figure saved to {outname}")